In [ ]:
import os

In [ ]:
os.chdir('/home/witoslaw/diffTSS')
os.getcwd()

In [ ]:
from EPInformer.models import EPInformer_v2, enhancer_predictor_256bp
from scripts.utils import prepare_input, prepare_hd5_input
from sklearn.preprocessing import StandardScaler
import scripts.utils_forTraining as train
from scripts.utils import FastaStringExtractor, one_hot_encode
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from tqdm import tqdm
import torch
from torch.utils.data import Subset, Dataset
import h5py
import kipoiseq

In [ ]:
def create_h5_data(file_path, ensid_data, pe_code_data, distance_data, activity_data, hic_data):

  # Create a new H5 file
  with h5py.File(file_path, 'w') as h5_file:
      # Create 'ensid' dataset (an array of encoded strings)
      #ensid_data = [f"ENSG{100000 + i}" for i in range(10)]  # Example ENSIDs
      encoded_ensid_data = [s.encode() for s in ensid_data]  # Encode strings as bytes
      h5_file.create_dataset('ensid', data=np.array(encoded_ensid_data, dtype='S'))

      # Create 'pe_code' dataset (example: 2D array of integers)
      #pe_code_data = np.random.randint(0, 100, size=(10, 20))  # Replace with real data
      h5_file.create_dataset('pe_code', data=pe_code_data, dtype="float32")

      # Create 'distance' dataset (example: 2D array with distances)
      #distance_data = np.random.rand(10, 5)  # Replace with real distances
      h5_file.create_dataset('distance', data=distance_data)

      # Create 'activity' dataset (example: 2D array for enhancer activity)
      #activity_data = np.random.rand(10, 5)  # Replace with real activity data
      h5_file.create_dataset('activity', data=activity_data)

      # Create 'hic' dataset (example: 2D array for Hi-C contact frequencies)
      #hic_data = np.random.rand(10, 5)  # Replace with real Hi-C data
      h5_file.create_dataset('hic', data=hic_data)

  print(f"H5 file created at {file_path}")

In [ ]:
os.listdir('./data')

In [ ]:
data = h5py.File('data/K562_enhancer_promoter_encoding.rna_embedding.hg38.h5')

In [ ]:
data.keys()

In [9]:
for key in data.keys():
    print(data[key].shape)

(29077, 61)
(29077, 61)
(29077,)
(29077, 61)
(29077, 61, 2000, 5)


In [10]:
np.array(data['ensid'])

array([b'ENSG00000310526', b'ENSG00000225630', b'ENSG00000237973', ...,
       b'ENSG00000187191', b'ENSG00000205916', b'ENSG00000252426'],
      dtype='|S18')

In [ ]:
bed_files = {}
for file in [x for x in os.listdir('data/RNASeq_bw') if x.endswith('.bed') and 'TSS' not in x]:
    bed_files[f"{file.split('.')[0]}.{file.split('.')[2]}"] = pd.read_csv(f'data/RNASeq_bw/{file}', sep='\t')

In [12]:
bed_files.keys()

dict_keys(['K562.ENCFF777EAJ', 'GM12878.ENCFF546NVF', 'GM12878.ENCFF164VLA', 'GM12878.ENCFF985TNZ', 'GM12878.ENCFF143BSQ', 'GM12878.ENCFF074SXQ', 'GM12878.ENCFF155XJQ', 'K562.ENCFF528VFJ', 'GM12878.ENCFF078ATR', 'GM12878.ENCFF892WMR', 'GM12878.ENCFF037DUE', 'GM12878.ENCFF182LTN', 'GM12878.ENCFF104OTO', 'GM12878.ENCFF395OHI', 'K562.ENCFF829PNJ', 'K562.ENCFF964BAP', 'K562.ENCFF006DQI', 'K562.ENCFF040DXX', 'K562.ENCFF097ASF', 'K562.ENCFF336COA', 'K562.ENCFF448XCV', 'K562.ENCFF451LAH', 'K562.ENCFF610CIC', 'K562.ENCFF990XSY'])

In [13]:
bed_files_genes = []
for bed in bed_files:
    bed_files_genes.append(bed_files[bed]['name'].astype('|S18').to_numpy())

In [23]:
gene_list_k562 = pd.read_csv('data/ABC-multiTSS_nominated/K562/Neighborhoods/GeneList.txt', sep='\t')
gene_list = list(gene_list_k562['Ensembl_ID'])
gene_list[0]

'ENSG00000310526'

In [15]:
for bed_genes in bed_files_genes:
    print(f"{(bed_genes == np.array(data['ensid'])).all()} - {(bed_genes == gene_list_k562['Ensembl_ID'].astype('|S18').to_numpy()).all()}")

True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True
True - True


In [ ]:
bin_16_files = {}
for file in [x for x in os.listdir('data/RNASeq_bw') if x.endswith('values_TSS.tab')]:
    bin_16_files[f"{file.split('.')[0]}.{file.split('.')[2]}"] = pd.read_csv(f'data/RNASeq_bw/{file}', sep='\t', skiprows=3, header=None)

In [ ]:
bin_16_files.keys()

In [ ]:
rna = np.loadtxt('data/RNASeq_bw/GM12878.minus.ENCFF078ATR_values_TSS.tab', delimiter='\t', skiprows=3)
rna.shape

In [ ]:
bin_16_files['GM12878.ENCFF074SXQ_values_TSS'].to_numpy()[0]

In [ ]:
scaler = StandardScaler()
scaler.fit(bin_16_files['GM12878.ENCFF074SXQ_values_TSS'].to_numpy())
standardized_data = scaler.transform(bin_16_files['GM12878.ENCFF074SXQ_values_TSS'].to_numpy())
standardized_data.max(axis=1)

In [33]:
enhancer_gene_k562_100kb = pd.read_csv('./data/K562_enhancer_gene_links_100kb.hg38.tsv', sep='\t')
gene_k562_tss = pd.read_csv('./data/ABC-multiTSS_nominated/K562/Neighborhoods/GeneList.txt', sep='\t')[['name', 'Ensembl_ID', 'chr', 'tss', 'strand', 'H3K27ac.RPM.TSS1Kb', 'DHS.RPM.TSS1Kb']]
gene_k562_tss['ENSID'] = gene_k562_tss['Ensembl_ID']


enhancer_gene_gm12878_100kb = pd.read_csv('./data/GM12878_enhancer_gene_links_100kb.hg38.tsv', sep='\t')
gene_gm12878_tss = pd.read_csv('./data/ABC-multiTSS_nominated/GM12878/Neighborhoods/GeneList.txt', sep='\t')[['name', 'Ensembl_ID', 'chr', 'tss', 'strand', 'H3K27ac.RPM.TSS1Kb', 'DHS.RPM.TSS1Kb']]
gene_gm12878_tss['ENSID'] = gene_gm12878_tss['Ensembl_ID']


enhancer_gene_k562_100kb_includeNoEnhancerGene = enhancer_gene_k562_100kb.merge(gene_k562_tss, left_on='TargetGeneEnsembl_ID', right_on='Ensembl_ID', how='right', suffixes=['', '_gene']).reset_index()
enhancer_gene_gm12878_100kb_includeNoEnhancerGene = enhancer_gene_gm12878_100kb.merge(gene_gm12878_tss, left_on='TargetGeneEnsembl_ID', right_on='Ensembl_ID', how='right', suffixes=['', '_gene']).reset_index()

gene_list = list(gene_k562_tss['ENSID'])


rna_df_K562 = pd.read_csv('./data/RNASeq_bw/K562.minus.ENCFF528VFJ_values_TSS.tab', header=None, sep='\t', skiprows=3)
rna_GM12878 = standardized_data

In [34]:
gene_enhancer_table = enhancer_gene_gm12878_100kb_includeNoEnhancerGene
promoter_signals = gene_gm12878_tss
cells = 'GM12878'
num_features=3
rna_encoding = False
rna_embedding = True
rna_df = rna_GM12878
#ensid_data, pe_codrna_GM12878e, distance_data, activity_data, hic_data = prepare_hd5_input(enhancer_gene_k562_100kb_includeNoEnhancerGene, gene_k562_tss, gene_list, 'K562', num_features=3, rna_encoding=True, rna_df=rna_df_K562)

In [ ]:
gene_enhancer_table

In [35]:
#def prepare_hd5_input(gene_enhancer_table, promoter_signals, gene_list, cells, num_features = 3, rna_encoding=False, rna_df=None):
mRNA_feauture = pd.read_csv('./data/RNA_CAGE.txt', sep='\t', index_col='ENSID')
promoter_signals['PromoterActivity'] = np.sqrt(promoter_signals['H3K27ac.RPM.TSS1Kb']*promoter_signals['DHS.RPM.TSS1Kb'])
promoter_signals.set_index('ENSID', inplace=True)
mRNA_feats = ['UTR5LEN_log10zscore',
   'CDSLEN_log10zscore', 'INTRONLEN_log10zscore', 'UTR3LEN_log10zscore',
   'UTR5GC', 'CDSGC', 'UTR3GC', 'ORFEXONDENSITY']
PE_code_list = []
#PE_feat_list = []
PE_distance_list = []
PE_activity_list = []
PE_contact_list = []
mRNA_promoter_list = []
PE_links_list = []
gene = gene_list[0]
gene_df = gene_enhancer_table[gene_enhancer_table['ENSID'] == gene]
if rna_encoding:
    gene_rna_df = rna_df[rna_df[3] == gene]
if rna_embedding:
    gene_rna_df = rna_df[gene_list.index(gene)]

In [128]:
batch_size = 16
random = torch.rand(batch_size, 128, 61, 125)

In [129]:
random.shape

torch.Size([16, 128, 61, 125])

In [132]:
rna_df = rna_dfs[0:61]
rna_df = np.array(rna_df)
rna_df = np.tile(rna_df, (batch_size, 1, 1))
rna_df = torch.from_numpy(rna_df)
rna_df.shape

torch.Size([16, 61, 125])

In [168]:
conc = torch.concat([random, torch.from_numpy(np.array(rna_dfs)).unsqueeze(1)], axis=1)

In [180]:
conc.shape

torch.Size([16, 129, 61, 125])

In [178]:
gene_enhancer_df = gene_df
fasta_path = './data/hg38.fa'
max_n_enhancer = 60
max_distanceToTSS = 100_000
max_seq_len=2000
add_flanking=False
rna_encoding=False
rna_embedding=True
rna_df=gene_rna
rna_dfs = []
for gene in tqdm(gene_list[0:batch_size]):
    gene_df = gene_enhancer_table[gene_enhancer_table['ENSID'] == gene]
    if rna_encoding:
        gene_rna_df = rna_df[rna_df[3] == gene]
    if rna_embedding:
        gene_rna_df = rna_GM12878[gene_list.index(gene)]
    PE_code, activity_list, distance_list, contact_list, gene_name, PE_links, rna_df = encode_promoter_enhancer_links(
        gene_df, 
        max_seq_len=2000, 
        max_n_enhancer=60, 
        max_distanceToTSS=100_000, 
        add_flanking=False, 
        rna_encoding=rna_encoding, 
        rna_embedding=rna_embedding,
        rna_df=gene_rna_df
    )
    
    rna_dfs.append(rna_df)


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:01<00:00, 14.82it/s]


In [179]:
np.array(rna_dfs).shape

(16, 61, 125)

In [177]:
def encode_promoter_enhancer_links(gene_enhancer_df, fasta_path = './data/hg38.fa', max_n_enhancer = 60, max_distanceToTSS = 100_000, max_seq_len=2000, add_flanking=False, rna_encoding=False, rna_embedding=False, rna_df=None):
    fasta_extractor = FastaStringExtractor(fasta_path)
    gene_pe = gene_enhancer_df.sort_values(by='distance')
    row_0 = gene_pe.iloc[0]
    gene_ensid=row_0['TargetGeneEnsembl_ID']
    gene_name = row_0['TargetGene']
    gene_tss = row_0['TargetGeneTSS']
    chrom = row_0['chr']
    if row_0['TargetGeneTSS'] != row_0['TargetGeneTSS']:
        gene_tss = row_0['tss']
        gene_name = row_0['name_gene']
        chrom = row_0['chr_gene']
    target_interval = kipoiseq.Interval(chrom, int(gene_tss-max_seq_len/2), int(gene_tss+max_seq_len/2))
    promoter_seq = fasta_extractor.extract(target_interval)
    promoter_code = one_hot_encode(promoter_seq)
    if rna_encoding:
        rna_df = rna_df[(rna_df[7] >= target_interval.start) & (rna_df[8] <= target_interval.end)]
        new_index = rna_df[7].values - target_interval.start
        #print(f'For GeneID {gene_ensid}: \nrna_df[9] with new index: \n{rna_df[[9]].set_index(new_index)} \n')
        #print(f'Unique indices of rna_df[9]: \n{np.unique(rna_df[[9]].set_index(new_index).index)}\nwith length {len(np.unique(rna_df[[9]].set_index(new_index).index))}')
        #print(f'And reindexed:\n{rna_df[[9]].set_index(new_index).reindex(list(range(0,max_seq_len)), fill_value=0)}')
        promoter_code = np.concatenate((promoter_code, rna_df[[9]].set_index(new_index).reindex(list(range(0,max_seq_len)), fill_value=0)), axis=1)
    if rna_embedding:
        rna_df = np.concatenate([rna_df.reshape(1, 125), np.zeros([60, 125])])
    enhancers_code = np.zeros((max_n_enhancer, max_seq_len, 4))
    enhancer_activity = np.zeros(max_n_enhancer)
    enhancer_distance = np.zeros(max_n_enhancer)
    enhancer_contact = np.zeros(max_n_enhancer)
    # set distance threshold
    gene_pe = gene_pe[(gene_pe['distance'] > max_seq_len/2)&(gene_pe['distance'] <= max_distanceToTSS)]
    e_i = 0
    gene_element_pair = []
    for idx, row in gene_pe.iterrows():
        if row['TargetGene'] != row['TargetGene']:
            break
        if pd.isna(row['start']):
            continue
        if e_i >= max_n_enhancer:
            break
        enhancer_start = int(row['start'])
        enhancer_end = int(row['end'])
        enhancer_center = int((row['start'] + row['end'])/2)
        enhancer_len = enhancer_end - enhancer_start
        # put sequence at the center
        if add_flanking:
            enhancer_target_interval = kipoiseq.Interval(chrom, enhancer_center-int(max_seq_len/2), enhancer_center+int(max_seq_len/2))
            enhancers_code[e_i][:] = one_hot_encode(fasta_extractor.extract(enhancer_target_interval))
        else:
            # enhancers_signal = np.zeros((max_n_enhancer, max_seq_len))
            if enhancer_len > max_seq_len:
                enhancer_target_interval = kipoiseq.Interval(chrom, enhancer_center-int(max_seq_len/2), enhancer_center+int(max_seq_len/2))
                enhancers_code[e_i][:] = one_hot_encode(fasta_extractor.extract(enhancer_target_interval))
            else:
                code_start = int(max_seq_len/2)-int(enhancer_len/2)
                enhancer_target_interval = kipoiseq.Interval(chrom, enhancer_start, enhancer_end)
                enhancers_code[e_i][code_start:code_start+enhancer_len] = one_hot_encode(fasta_extractor.extract(enhancer_target_interval))
        # put sequence from the start
        enhancer_activity[e_i] = row['activity_base']
        enhancer_distance[e_i] = row['distance']
        enhancer_contact[e_i] = row['hic_contact']
        gene_element_pair.append([gene_name, row['name']])
        e_i += 1
    # print(promoter_signals.shape, enhancers_signal.shape)
    if rna_encoding:
        enhancers_code = np.concatenate((enhancers_code, np.zeros((max_n_enhancer, max_seq_len, 1))), axis=2)
        pe_code = np.concatenate([promoter_code[np.newaxis,:], enhancers_code], axis=0, dtype=np.float32)
    else:
        pe_code = np.concatenate([promoter_code[np.newaxis,:], enhancers_code], axis=0)
    gene_element_pair = pd.DataFrame(gene_element_pair, columns=['gene', 'element'])
    if rna_embedding:
        return pe_code, enhancer_activity, enhancer_distance, enhancer_contact, gene_name, gene_element_pair, rna_df
    else:
        return pe_code, enhancer_activity, enhancer_distance, enhancer_contact, gene_name, gene_element_pair

In [ ]:
enhancer_gene_k562_100kb = pd.read_csv('./data/K562_enhancer_gene_links_100kb.hg38.tsv', sep='\t')
gene_k562_tss = pd.read_csv('./data/ABC-multiTSS_nominated/K562/Neighborhoods/GeneList.txt', sep='\t')[['name', 'Ensembl_ID', 'chr', 'tss', 'strand', 'H3K27ac.RPM.TSS1Kb', 'DHS.RPM.TSS1Kb']]
gene_k562_tss['ENSID'] = gene_k562_tss['Ensembl_ID']


enhancer_gene_gm12878_100kb = pd.read_csv('./data/GM12878_enhancer_gene_links_100kb.hg38.tsv', sep='\t')
gene_gm12878_tss = pd.read_csv('./data/ABC-multiTSS_nominated/GM12878/Neighborhoods/GeneList.txt', sep='\t')[['name', 'Ensembl_ID', 'chr', 'tss', 'strand', 'H3K27ac.RPM.TSS1Kb', 'DHS.RPM.TSS1Kb']]
gene_gm12878_tss['ENSID'] = gene_gm12878_tss['Ensembl_ID']


enhancer_gene_k562_100kb_includeNoEnhancerGene = enhancer_gene_k562_100kb.merge(gene_k562_tss, left_on='TargetGeneEnsembl_ID', right_on='Ensembl_ID', how='right', suffixes=['', '_gene']).reset_index()
enhancer_gene_gm12878_100kb_includeNoEnhancerGene = enhancer_gene_gm12878_100kb.merge(gene_gm12878_tss, left_on='TargetGeneEnsembl_ID', right_on='Ensembl_ID', how='right', suffixes=['', '_gene']).reset_index()

gene_list = list(gene_k562_tss['ENSID'])


rna_df_K562 = pd.read_csv('./data/RNASeq_bw/K562.minus.ENCFF528VFJ.coverage.dedup.txt', header=None, sep='\t')
rna_df_GM12878 = pd.read_csv('./data/RNASeq_bw/GM12878.minus.ENCFF074SXQ.coverage.dedup.txt', header=None, sep='\t')


ensid_data, pe_code, distance_data, activity_data, hic_data = prepare_hd5_input(enhancer_gene_k562_100kb_includeNoEnhancerGene, gene_k562_tss, gene_list, 'K562', num_features=3, rna_encoding=True, rna_df=rna_df_K562)
np.save('K562.ensid.rna_encoding.npy', ensid_data)
np.save('K562.pe_code.rna_encoding.npy', pe_code)
np.save('K562.distance.rna_encoding.npy', distance_data)
np.save('K562.activity.rna_encoding.npy', activity_data)
np.save('K562.hic.rna_encoding.npy', hic_data)

file_path = '/scratch/han_lab/dwito/EPInformer/K562_enhancer_promoter_encoding.rna_encoding.hg38.h5'
create_h5_data(file_path, ensid_data, pe_code, distance_data, activity_data, hic_data)